# Homework 4: using MLFlow with Spark MLlib

In this homework, we use MLFlow to track model training in Spark MLlib

## Dataset Review

The Adult dataset is publicly available at the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Adult).
This data derives from census data and consists of information about 48842 individuals and their annual income.
You can use this information to predict if an individual earns **<=50K or >50k** a year.
The dataset consists of both numeric and categorical variables.

Attribute Information:

- age: continuous
- workclass: Private,Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked
- fnlwgt: continuous
- education: Bachelors, Some-college, 11th, HS-grad, Prof-school, Assoc-acdm, Assoc-voc...
- education-num: continuous
- marital-status: Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent...
- occupation: Tech-support, Craft-repair, Other-service, Sales, Exec-managerial, Prof-specialty, Handlers-cleaners...
- relationship: Wife, Own-child, Husband, Not-in-family, Other-relative, Unmarried
- race: White, Asian-Pac-Islander, Amer-Indian-Eskimo, Other, Black
- sex: Female, Male
- capital-gain: continuous
- capital-loss: continuous
- hours-per-week: continuous
- native-country: United-States, Cambodia, England, Puerto-Rico, Canada, Germany...

Target/Label: - <=50K, >50K


## Load Data

The Adult dataset is available in databricks-datasets. Read in the data using the CSV data source for Spark and rename the columns appropriately.


### 1. list data files in DBFS directory `/databricks-datasets/adult`

> hint: use `!ls` or `dbutils.fs.ls()`

In [ ]:
!ls /databricks-datasets/adult

adult.data  adult.test	README.md


### 2. View some sample rows from `adult.data` to verify the format of the data
> hint: use `!head` or `dbutils.fs.head()`

In [ ]:
!head /databricks-datasets/adult/adult.data

39, State-gov, 77516, Bachelors, 13, Never-married, Adm-clerical, Not-in-family, White, Male, 2174, 0, 40, United-States, <=50K
50, Self-emp-not-inc, 83311, Bachelors, 13, Married-civ-spouse, Exec-managerial, Husband, White, Male, 0, 0, 13, United-States, <=50K
38, Private, 215646, HS-grad, 9, Divorced, Handlers-cleaners, Not-in-family, White, Male, 0, 0, 40, United-States, <=50K
53, Private, 234721, 11th, 7, Married-civ-spouse, Handlers-cleaners, Husband, Black, Male, 0, 0, 40, United-States, <=50K
28, Private, 338409, Bachelors, 13, Married-civ-spouse, Prof-specialty, Wife, Black, Female, 0, 0, 40, Cuba, <=50K
37, Private, 284582, Masters, 14, Married-civ-spouse, Exec-managerial, Wife, White, Female, 0, 0, 40, United-States, <=50K
49, Private, 160187, 9th, 5, Married-spouse-absent, Other-service, Not-in-family, Black, Female, 0, 0, 16, Jamaica, <=50K
52, Self-emp-not-inc, 209642, HS-grad, 9, Married-civ-spouse, Exec-managerial, Husband, White, Male, 0, 0, 45, United-States, >50K
31, 

### 3. Load data using Spark with the following schema:

```
schema =  """age double, workclass string, fnlwgt double, education string, education_num double, 
marital_status string, occupation string, relationship string, race string, sex string,
capital_gain double, capital_loss double, hours_per_week double, native_country string, income string"""
```


In [ ]:
schema = """age double, workclass string, fnlwgt double, education string, 
education_num double, marital_status string, occupation string, relationship string, 
race string, sex string, capital_gain double, capital_loss double, 
hours_per_week double, native_country string, income string"""

In [ ]:
df = spark.read.csv(
    "/databricks-datasets/adult/adult.data",
    schema=schema,
    header=False
)

### 4. Split the data into train/test using 80/20 split, using a seed of 42. 

Display the number of records in each dataframe.

Tip:
```
DF.randomSplit(weights=[w1,w2,...], seed=xx)
```

In [ ]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("Train count:", train_df.count())
print("Test count:", test_df.count())

Train count: 26242
Test count: 6319


## Preprocess Data


### 5. In the following, build a feature pipeline that does the following:

- First index string columns `["workclass", "education", "marital_status", "occupation", "relationship", "race", "sex", "native_country"]` using a string indexer
- Then encoding the indexed strings using `OneHotEncoder`
- Assemble the encoded columns into a feature vector "features"
- use a `StringIndexer` to index `income` (our outcome variable)  and call it `label`


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder


In [ ]:
cat_cols = ["workclass", "education", "marital_status", "occupation",
            "relationship", "race", "sex", "native_country"]

In [ ]:
# Index categorical features
indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in cat_cols
]

# One-hot encode
encoders = [
    OneHotEncoder(inputCol=c + "_idx", outputCol=c + "_ohe")
    for c in cat_cols
]

# Label indexer
label_indexer = StringIndexer(inputCol="income", outputCol="label")

# Assemble features
feature_cols = [c + "_ohe" for c in cat_cols] + [
    "age", "fnlwgt", "education_num", "capital_gain",
    "capital_loss", "hours_per_week"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

pipeline = Pipeline(stages=indexers + encoders + [label_indexer, assembler])

### 6. Train the Pipeline using `train`. 

- Then apply the trained pipeline on `train`/`test` to obtain `train_preped` and `test_preped`.

In [ ]:
pipeline_model = pipeline.fit(train_df)

train_preped = pipeline_model.transform(train_df)
test_preped = pipeline_model.transform(test_df)

## Fit and Evaluate Models

Now, try out some of the Logistic Regression algorithm with MLflow tracking.




### 7. Import pyspark and mlflow

- import `pyspark`, `pyspark.ml`, and `mlflow`


> Note: we did not enable autolog because on the Free Edition, the autolog does not seem to work well with pyspark.ml. Instead we will manually log key metrics and parameters as an illustration.

In [ ]:
import pyspark
import pyspark.ml
import mlflow

### 8. trains a logistic regression model `lrModel` and save the model run as `'logistic'` in mlflow.

- Set maxIter=10. 
- Then use the model to make predictions on the test data 
- Calculate `areaUnderROC` and `accuracy` metrics for the predicted outcomes 
- log `areaUnderROC` as `auc`
- log `accuracy` as `accuracy`
- log the model's parameters including 
  * `MaxIter (using [model].getMaxIter())`
  * `RegParam (using [model].getMaxIter())`
  * `ElasticNetParam (using [model].getElasticNetParam())`
- print AUC and accuracy

> Note: areaUnderROC and accuracy belong to BinaryClassificationEvaluator and MulticlassClassificationEvaluator respectively. 

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator


In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import mlflow

with mlflow.start_run(run_name="logistic"):

    # Train model
    lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)
    lrModel = lr.fit(train_preped)

    # Make predictions
    preds = lrModel.transform(test_preped)

    # Evaluators
    auc_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
    acc_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")

    auc = auc_eval.evaluate(preds)
    acc = acc_eval.evaluate(preds)

    # Log metrics
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("accuracy", acc)

    # Log parameters (IMPORTANT: use lr, NOT lrModel)
    mlflow.log_param("maxIter", lr.getMaxIter())
    mlflow.log_param("regParam", lr.getRegParam())
    mlflow.log_param("elasticNetParam", lr.getElasticNetParam())

    # Print results
    print("AUC:", auc)
    print("Accuracy:", acc)

AUC: 0.9035865107899003
Accuracy: 0.8539325842696629


### 9. View model's predictions and probabilities of each prediction class.  

- Display these columns: `("label", "prediction", "probability", "age", "occupation")`

In [ ]:
preds.select(
    "label",
    "prediction",
    "probability",
    "age",
    "occupation"
).show(20, truncate=False)

+-----+----------+------------------------------------------+----+------------------+
|label|prediction|probability                               |age |occupation        |
+-----+----------+------------------------------------------+----+------------------+
|0.0  |0.0       |[0.9998426014237732,1.573985762267638E-4] |17.0| ?                |
|0.0  |0.0       |[0.9995409183533678,4.5908164663222006E-4]|17.0| ?                |
|0.0  |0.0       |[0.9998603767837044,1.3962321629557017E-4]|17.0| ?                |
|0.0  |0.0       |[0.999532789284524,4.6721071547595194E-4] |17.0| Prof-specialty   |
|0.0  |0.0       |[0.9996795361672284,3.204638327716225E-4] |17.0| Farming-fishing  |
|0.0  |0.0       |[0.9997902458185799,2.097541814201076E-4] |17.0| Other-service    |
|0.0  |0.0       |[0.9991127066066564,8.872933933435911E-4] |17.0| Farming-fishing  |
|0.0  |0.0       |[0.999594056999406,4.0594300059404453E-4] |17.0| Sales            |
|0.0  |0.0       |[0.9968535722659503,0.00314642773404

## Tracking Cross Validation

### 10. Define a cross validator that 
- use logistic regression as an estimator (maxIter=10)
- does 3-folds cross-validation
- explore the following parameter values
- using areaUnderROC as the metric

```
  regParam: [0.01, 0.5, 2.0])
  elasticNetParam: [0.0, 0.5, 1.0])
```

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Base model
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)

# Parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.5, 2.0]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

# Evaluator (AUC)
evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

# CrossValidator
cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3
)


### 11. Create a volume for this homework and set caching path for the crossvalidator

Run the following to set up the volume for the assignment and use it as the caching path for the CrossValidator

In [ ]:
%sql
create catalog if not exists msbabigdata;
create schema if not exists msbabigdata.spark;
create volume if not exists msbabigdata.spark.homework4;

In [ ]:
import os

# Define your UC Volume path
# Format: /Volumes/<catalog>/<schema>/<volume>/<optional_path>
uc_volume_path = "/Volumes/msbabigdata/spark/homework4"

# Set the environment variable for SparkML caching
os.environ['SPARKML_TEMP_DFS_PATH'] = uc_volume_path



### 12. Start a new MLFlow run (`run_name='logistic_cv'`). 

In this run:

- fit the crossvalidator on the training data
- use the fitted model to make predictions for the test data
- Calculate areaUnderROC and accuracy metrics for the predicted outcomes
- log areaUnderROC as auc
- log accuracy as accuracy
- log the model's parameters including
  - MaxIter (using [model].getMaxIter())
  - RegParam (using [model].getMaxIter())
  - ElasticNetParam (using [model].getElasticNetParam())
- print AUC and accuracy

> Tip: to get the logistic regression model chosen by the CrossValidator, use the trained crossvalidator's `.bestModel`

In [ ]:
import mlflow
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Rebuild base model
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)

# Reuse your existing paramGrid + evaluator
evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3
)

with mlflow.start_run(run_name="logistic_cv"):

    cvModel = cv.fit(train_preped)

    # FORCE Spark model type
    bestModel = cvModel.bestModel

    preds = bestModel.transform(test_preped)

    auc = evaluator.evaluate(preds)
    acc = MulticlassClassificationEvaluator(
        labelCol="label",
        metricName="accuracy"
    ).evaluate(preds)

    mlflow.log_metric("auc", auc)
    mlflow.log_metric("accuracy", acc)

    # IMPORTANT: safe extraction (Spark internal JVM model)
    mlflow.log_param("maxIter", bestModel.getMaxIter())
    mlflow.log_param("regParam", bestModel.getRegParam())
    mlflow.log_param("elasticNetParam", bestModel.getElasticNetParam())

    print("AUC:", auc)
    print("Accuracy:", acc)

AUC: 0.903584336776427
Accuracy: 0.8539325842696629


In [ ]:
print(type(bestModel))

<class 'pyspark.ml.classification.LogisticRegressionModel'>


### 13. Choose the model using MLflow dashboard

Compared the two runs that we have tracked, which model would you choose? why, and what's the chosen model's key parameters including `RegParam` and `ElasticNetParam`?

I would choose the logistic_cv model because it uses 3-fold cross-validation to systematically search over 9 hyperparameter combinations (regParam × elasticNetParam), selecting the best-performing configuration. This makes it more reliable and generalizable than the plain logistic model, which simply used Spark's defaults (regParam=0.0, elasticNetParam=0.0) with no tuning.

While both models achieved nearly identical test performance (AUC = 0.9036, Accuracy = 0.8539), the logistic_cv result is more trustworthy cross-validation reduces the risk of overfitting to a single train/test split, meaning the model is more likely to maintain its performance on new, unseen data.

Key parameters of the chosen logistic_cv model:

RegParam: 0.01, minimal regularization, suggesting the model benefits from fitting the data closely without heavy penalization
ElasticNetParam: 0.0, pure L2 (Ridge) regularization, meaning all features are retained with small, shrunk coefficients rather than being zeroed out
maxIter: 10